In [8]:
import zipfile
with zipfile.ZipFile('ml-latest-small.zip', 'r') as zip_ref:
  zip_ref.extractall('data')


In [14]:
# import the dataset
import pandas as pd
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

In [ ]:
print('The dimensions of movies dataframe are:', movies_df.shape, '\nThe dimensions of ratings dataframe are:', ratings_df.shape)
print('')

The dimensions of movies dataframe are: (9742, 3) 
The dimensions of ratings dataframe are: (100836, 4)


In [16]:
# Take a look at movies dataframe
movies_df.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
# Take a look at ratings dataframe
ratings_df.head()

In [18]:
# Movie ID to movie name mapping
movie_names = movies_df.set_index('movieId')['title'].to_dict()
# Count unique users and movies
n_users = len(ratings_df.userId.unique())
n_items = len(ratings_df.movieId.unique())
print("Number of unique users:", n_users)
print("Number of unique movies:", n_items)
# Total matrix size
print("The full rating matrix will have:", n_users * n_items, "elements.")
print('---')
# Number of ratings
print("Number of ratings:", len(ratings_df))
print("Therefore: ", len(ratings_df) / (n_users*n_items) * 100, '% of the matrix is filled.')
print("We have an incredibly sparse matrix to work with here.")
print("And ... as you can imagine, as the number of users and products grow, the number of elements will increase by n*2")
print("You are going to need a lot of memory to work with global scale ... storing a full matrix in memory would be a challenge.")
print("One advantage here is that matrix factorization can realize the rating matrix implicitly, thus we don't need all the data")


Number of unique users: 610
Number of unique movies: 9724
The full rating matrix will have: 5931640 elements.
---
Number of ratings: 100836
Therefore:  1.6999683055613624 % of the matrix is filled.
We have an incredibly sparse matrix to work with here.
And ... as you can imagine, as the number of users and products grow, the number of elements will increase by n*2
You are going to need a lot of memory to work with global scale ... storing a full matrix in memory would be a challenge.
One advantage here is that matrix factorization can realize the rating matrix implicitly, thus we don't need all the data


In [19]:
import torch
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader

# Creating the dataloader (necessary for PyTorch)

class Loader(Dataset):

    def __init__(self):
        self.ratings = ratings_df.copy()

        # Extract all user IDs and movie IDs
        users = ratings_df.userId.unique()
        movies = ratings_df.movieId.unique()

        # Producing new continuous IDs for users and movies
        self.userid2idx = {o: i for i, o in enumerate(users)}
        self.movieid2idx = {o: i for i, o in enumerate(movies)}

        # Reverse mappings
        self.idx2userid = {i: o for o, i in self.userid2idx.items()}
        self.idx2movieid = {i: o for o, i in self.movieid2idx.items()}

        # Replace original IDs with indexed IDs
        self.ratings.movieId = ratings_df.movieId.apply(
            lambda x: self.movieid2idx[x]
        )

        self.ratings.userId = ratings_df.userId.apply(
            lambda x: self.userid2idx[x]
        )

        # Features and targets
        self.x = self.ratings.drop(
            ['rating', 'timestamp'],
            axis=1
        ).values

        self.y = self.ratings['rating'].values

        # Transforms the data to tensors (ready for torch models.)
        self.x = torch.tensor(self.x)
        self.y = torch.tensor(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.ratings)

In [ ]:
import torch
import torch.nn as nn

class MatrixFactorization(nn.Module):

    def __init__(self, n_users, n_items, n_factors=20):
        super().__init__()

        # User embeddings
        self.user_factors = nn.Embedding(
            n_users,
            n_factors
        )

        # Movie embeddings
        self.item_factors = nn.Embedding(
            n_items,
            n_factors
        )

        # Initialize weights
        self.user_factors.weight.data.uniform_(0, 0.05)
        self.item_factors.weight.data.uniform_(0, 0.05)

    def forward(self, data):

        # Extract user and movie IDs
        users = data[:, 0]
        items = data[:, 1]

        # Get embeddings
        user_embedding = self.user_factors(users)
        item_embedding = self.item_factors(items)

        # Dot product
        predictions = (
            user_embedding * item_embedding
        ).sum(1)

        return predictions

In [25]:
num_epochs = 128

# Check if GPU is available
cuda = torch.cuda.is_available()

print("Is running on GPU:", cuda)

# Initialize model
model = MatrixFactorization(
    n_users,
    n_items,
    n_factors=8
)

print(model)

# Print model parameters
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.data)

# Move model to GPU if available
if cuda:
    model = model.cuda()

# MSE Loss
loss_fn = torch.nn.MSELoss()

# Adam Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# Training data
train_set = Loader()

train_loader = DataLoader(
    train_set,
    batch_size=128,
    shuffle=True
)

Is running on GPU: True
MatrixFactorization(
  (user_factors): Embedding(610, 8)
  (item_factors): Embedding(9724, 8)
)
user_factors.weight tensor([[0.0391, 0.0318, 0.0464,  ..., 0.0293, 0.0170, 0.0394],
        [0.0233, 0.0041, 0.0332,  ..., 0.0093, 0.0080, 0.0346],
        [0.0082, 0.0200, 0.0119,  ..., 0.0206, 0.0041, 0.0088],
        ...,
        [0.0450, 0.0247, 0.0396,  ..., 0.0269, 0.0306, 0.0200],
        [0.0164, 0.0105, 0.0217,  ..., 0.0016, 0.0454, 0.0478],
        [0.0429, 0.0023, 0.0383,  ..., 0.0331, 0.0481, 0.0202]])
item_factors.weight tensor([[2.0359e-02, 3.9455e-02, 2.9080e-02,  ..., 1.4431e-02, 7.2835e-03,
         1.4326e-02],
        [2.2355e-02, 3.0847e-02, 3.2653e-02,  ..., 2.6982e-02, 4.3725e-02,
         2.7193e-02],
        [5.4619e-03, 2.8900e-02, 7.8612e-03,  ..., 1.6004e-03, 1.0851e-02,
         1.9464e-05],
        ...,
        [2.5414e-02, 3.7841e-02, 4.2366e-02,  ..., 3.1249e-02, 3.5300e-03,
         3.6516e-02],
        [1.9438e-02, 4.7931e-02, 1.3196e-

In [26]:
from tqdm.notebook import tqdm

# Training loop
for it in tqdm(range(num_epochs)):

    losses = []

    for x, y in train_loader:

        # Move data to GPU
        if cuda:
            x = x.cuda()
            y = y.cuda()

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(x)

        # Calculate loss
        loss = loss_fn(
            outputs.squeeze(),
            y.type(torch.float32)
        )

        # Store loss
        losses.append(loss.item())

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

    # Print average loss per epoch
    print(
        "iter #{}".format(it),
        "Loss:",
        sum(losses) / len(losses)
    )

  0%|          | 0/128 [00:00<?, ?it/s]

iter #0 Loss: 11.064376784460194
iter #1 Loss: 4.744715572916312
iter #2 Loss: 2.474956035311452
iter #3 Loss: 1.7218885987543213
iter #4 Loss: 1.3466697175641955
iter #5 Loss: 1.1289949122873055
iter #6 Loss: 0.9919345218518059
iter #7 Loss: 0.9005824447737127
iter #8 Loss: 0.8374481561976641
iter #9 Loss: 0.7924791277589531
iter #10 Loss: 0.7594309207902947
iter #11 Loss: 0.7348527757800775
iter #12 Loss: 0.7162937288205635
iter #13 Loss: 0.7019028892447501
iter #14 Loss: 0.6904921570632059
iter #15 Loss: 0.6818333461048639
iter #16 Loss: 0.6748118990947147
iter #17 Loss: 0.6698469159945013
iter #18 Loss: 0.6659717742486049
iter #19 Loss: 0.6629175296440948
iter #20 Loss: 0.660848686386486
iter #21 Loss: 0.6590042736778404
iter #22 Loss: 0.6577947735030034
iter #23 Loss: 0.6568772635862307
iter #24 Loss: 0.6559732007571888
iter #25 Loss: 0.6553344995464165
iter #26 Loss: 0.6546319011731196
iter #27 Loss: 0.6538461747054521
iter #28 Loss: 0.6534579981326452
iter #29 Loss: 0.6522245952

In [27]:
# By training the model, we will have tuned latent factors for movies and users.

c = 0
uw = 0
iw = 0

for name, param in model.named_parameters():

    if param.requires_grad:

        print(name, param.data)

        if c == 0:
            uw = param.data
            c += 1

        else:
            iw = param.data

        # print('param_data', param.data)

user_factors.weight tensor([[ 0.8827,  1.0898,  1.3182,  ...,  2.0368,  1.1575,  1.6869],
        [ 1.0946,  0.9313,  0.3009,  ...,  1.7015, -0.3052,  1.7418],
        [ 0.5072,  0.5541, -1.1042,  ...,  2.4494, -2.1670, -0.0098],
        ...,
        [ 1.6564,  1.0319, -0.0847,  ...,  0.0254,  1.1524,  1.4156],
        [ 0.4410,  0.4323,  0.9842,  ...,  0.6522,  1.1605,  1.5645],
        [ 0.6503,  1.6079,  1.1084,  ...,  1.5076,  1.0742,  0.1859]],
       device='cuda:0')
item_factors.weight tensor([[0.3193, 0.6054, 0.6509,  ..., 0.3333, 0.3602, 0.2349],
        [0.8174, 0.3334, 0.3350,  ..., 0.5492, 0.7215, 0.2457],
        [0.5059, 0.6610, 0.3197,  ..., 0.3676, 0.5874, 0.3657],
        ...,
        [0.3487, 0.3628, 0.3652,  ..., 0.3548, 0.3283, 0.2752],
        [0.4074, 0.4426, 0.4063,  ..., 0.4190, 0.4369, 0.3409],
        [0.3774, 0.4086, 0.3944,  ..., 0.4000, 0.3733, 0.3098]],
       device='cuda:0')


In [29]:
trained_movie_embeddings = model.item_factors.weight.data.cpu().numpy()

In [31]:
trained_movie_embeddings # unique movie factor weights

array([[0.31931353, 0.6053821 , 0.6509027 , ..., 0.33327234, 0.36023465,
        0.23485565],
       [0.81737024, 0.3334382 , 0.33495826, ..., 0.54924613, 0.7214805 ,
        0.24569681],
       [0.50588316, 0.6610473 , 0.31974548, ..., 0.36761215, 0.58741397,
        0.36565262],
       ...,
       [0.3486699 , 0.3628002 , 0.36519903, ..., 0.35476258, 0.32833222,
        0.27515712],
       [0.40744776, 0.44258067, 0.4063129 , ..., 0.4189631 , 0.4369321 ,
        0.34094414],
       [0.37737662, 0.40863612, 0.3944228 , ..., 0.40002006, 0.37327468,
        0.30984274]], dtype=float32)

In [32]:
from sklearn.cluster import KMeans
# Fit the clusters based on the movie weights
kmeans = KMeans(n_clusters=10, random_state=0).fit(trained_movie_embeddings)

In [35]:
"""
It can be seen here that the movies in the same cluster tend to have
similar genres. Also note that the algorithm is unfamiliar with the movie names
and only obtained the relationships by looking at the numbers
representing how users responded to movie selections.
"""

import numpy as np

for cluster in range(10):

    print(f"\nCluster #{cluster}")

    movs = []

    for movidx in np.where(kmeans.labels_ == cluster)[0]:

        movid = train_set.idx2movieid[movidx]

        rat_count = ratings_df.loc[
            ratings_df['movieId'] == movid
        ].shape[0]

        movs.append(
            (movie_names[movid], rat_count)
        )

    for mov in sorted(
        movs,
        key=lambda tup: tup[1],
        reverse=True
    )[:10]:

        print("\t", mov[0])


Cluster #0
	 Waterworld (1995)
	 Back to the Future Part III (1990)
	 Back to the Future Part II (1989)
	 American President, The (1995)
	 Dragonheart (1996)
	 Three Musketeers, The (1993)
	 Beverly Hills Cop III (1994)
	 Wedding Crashers (2005)
	 Congo (1995)
	 Client, The (1994)

Cluster #1
	 Star Wars: Episode IV - A New Hope (1977)
	 Jurassic Park (1993)
	 Star Wars: Episode V - The Empire Strikes Back (1980)
	 Star Wars: Episode VI - Return of the Jedi (1983)
	 Aladdin (1992)
	 True Lies (1994)
	 Dark Knight, The (2008)
	 Pirates of the Caribbean: The Curse of the Black Pearl (2003)
	 Princess Bride, The (1987)
	 Finding Nemo (2003)

Cluster #2
	 Forrest Gump (1994)
	 Shawshank Redemption, The (1994)
	 Silence of the Lambs, The (1991)
	 Matrix, The (1999)
	 Braveheart (1995)
	 Terminator 2: Judgment Day (1991)
	 Schindler's List (1993)
	 Toy Story (1995)
	 Seven (a.k.a. Se7en) (1995)
	 Apollo 13 (1995)

Cluster #3
	 Mission: Impossible (1996)
	 GoldenEye (1995)
	 Avatar (2009)
	 